In [1]:
import cv2
import numpy as np
import pandas as pd

In [2]:
def calculate_video_metrics_with_tracking(data, blink_threshold=0.2, yawn_threshold=0.1):
    """
    Calculate various metrics from facial landmark data with progress tracking.

    Parameters:
        data (pd.DataFrame): DataFrame with columns ['frame', 'landmark_index', 'x_scaled', 'y_scaled'].

    Returns:
        dict: A dictionary containing aggregated metrics for the video.
    """
    print("Starting metric calculation process.")

    metrics = {
        "blink_count": 0,
        "mean_blink_duration": 0,
        "min_blink_duration": float('inf'),
        "max_blink_duration": 0,
        "std_blink_duration": 0,
        "mean_left_eye_openness": 0,
        "mean_right_eye_openness": 0,
        "average_smile_intensity": 0, 
        "max_smile_intensity": 0,
        "total_horizontal_movement": 0, 
        "total_vertical_movement": 0,
        "yawn_count": 0,
        "mean_yawn_duration": 0,
        "max_yawn_duration": 0,
        "std_yawn_duration": 0,        
        "y_tilt_mean": 0, 
        "y_tilt_max": 0, 
        "y_tilt_std": 0,
        "x_tilt_mean": 0, 
        "x_tilt_max": 0, 
        "x_tilt_std": 0,
    }

    left_eye_upper_indices = [466, 388, 387, 386, 385, 384, 398]
    left_eye_lower_indices = [263, 249, 390, 373, 374, 380, 381, 382, 362]
    left_eye_indices = left_eye_upper_indices + left_eye_lower_indices
    right_eye_upper_indices = [246, 161, 160, 159, 158, 157, 173]
    right_eye_lower_indices = [33, 7, 163, 144, 145, 153, 154, 155, 133]
    right_eye_indices = right_eye_upper_indices + right_eye_lower_indices
    lips_upper_outer= [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291]
    lips_lower_outer= [146, 91, 181, 84, 17, 314, 405, 321, 375, 291]
    lips_upper_inner= [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308]
    lips_lower_inner= [78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308]
    middle_line_indices= [10,9,6,4,1,152]
    nose_tip_index = 1


    def eye_aspect_ratio(eye_landmarks):
        # Compute distances between the vertical landmarks

        vertical_1 = np.linalg.norm(
            eye_landmarks.iloc[5][['x_scaled', 'y_scaled']].values - 
            eye_landmarks.iloc[9][['x_scaled', 'y_scaled']].values
        )
        vertical_2 = np.linalg.norm(
            eye_landmarks.iloc[3][['x_scaled', 'y_scaled']].values - 
            eye_landmarks.iloc[11][['x_scaled', 'y_scaled']].values
        )
        # Compute distance between the horizontal landmarks
        horizontal = np.linalg.norm(
            eye_landmarks.iloc[1][['x_scaled', 'y_scaled']].values - 
            eye_landmarks.iloc[2][['x_scaled', 'y_scaled']].values
        )

        return (vertical_1 + vertical_2) / (2.0 * horizontal)

    def compute_openness(eye_landmarks):
        vertical_1 = np.linalg.norm(
            eye_landmarks.iloc[5][['x_scaled', 'y_scaled']].values -
            eye_landmarks.iloc[9][['x_scaled', 'y_scaled']].values
        )
        vertical_2 = np.linalg.norm(
            eye_landmarks.iloc[3][['x_scaled', 'y_scaled']].values -
            eye_landmarks.iloc[11][['x_scaled', 'y_scaled']].values
        )
        return (vertical_1 + vertical_2) / 2

    def mouth_aspect_ratio(upper_lip_landmarks, lower_lip_landmarks):
        # Calculate the vertical distance between the upper and lower lip
        vertical_dist = np.linalg.norm(
            upper_lip_landmarks[['x_scaled', 'y_scaled']].mean().values -
            lower_lip_landmarks[['x_scaled', 'y_scaled']].mean().values
        )
        return vertical_dist


    blink_durations = []
    blink_timestamps = []
    is_blinking = False
    blink_start = None
    smile_intensities = []
    yawn_timestamps = []
    yawn_durations = []
    is_yawning = False
    yawn_start = None
    tilts_y = []
    tilts_x = []
    
    
    for frame, frame_data in data.groupby('frame'):
        if frame % (60*27) == 0:
            print(f"{frame} minute {frame/(60*27)}")
        
        eye_openness_per_frame = {"left_eye_openness": [], "right_eye_openness": []}
        
        left_eye = frame_data[frame_data['landmark_index'].isin(left_eye_indices)]
        right_eye = frame_data[frame_data['landmark_index'].isin(right_eye_indices)]
        upper_outer = frame_data[frame_data['landmark_index'].isin(lips_upper_outer)]
        lower_outer = frame_data[frame_data['landmark_index'].isin(lips_lower_outer)]
        upper_inner = frame_data[frame_data['landmark_index'].isin(lips_upper_inner)]
        lower_inner = frame_data[frame_data['landmark_index'].isin(lips_lower_inner)]
        y_coords = frame_data[frame_data['landmark_index'].isin(middle_line_indices)]['y_scaled'].values
        x_coords = frame_data[frame_data['landmark_index'].isin(middle_line_indices)]['x_scaled'].values
        
        if not left_eye.empty and not right_eye.empty:
            left_ear = eye_aspect_ratio(left_eye)
            right_ear = eye_aspect_ratio(right_eye)
            mean_ear = (left_ear + right_ear) / 2
            
            if mean_ear < blink_threshold:
                if not is_blinking:
                    is_blinking = True
                    blink_start = frame
            else:
                if is_blinking:
                    is_blinking = False
                    blink_durations.append(frame - blink_start)
                    blink_timestamps.append(frame/27)
        
                    
                    
        # Calculate openness if data for both eyes is available
        if not left_eye.empty:
            left_openness = compute_openness(left_eye)
            eye_openness_per_frame["left_eye_openness"].append(left_openness)
        else:
            eye_openness_per_frame["left_eye_openness"].append(None)

        if not right_eye.empty:
            right_openness = compute_openness(right_eye)
            eye_openness_per_frame["right_eye_openness"].append(right_openness)
        else:
            eye_openness_per_frame["right_eye_openness"].append(None)
            
            
            
        if not (upper_outer.empty or lower_outer.empty or upper_inner.empty or lower_inner.empty):
            # Horizontal distance (width of the smile)
            left_corner = frame_data[frame_data['landmark_index'] == lips_upper_outer[0]][['x_scaled', 'y_scaled']].values
            right_corner = frame_data[frame_data['landmark_index'] == lips_upper_outer[-1]][['x_scaled', 'y_scaled']].values
            if left_corner.size > 0 and right_corner.size > 0:
                horizontal_distance = np.linalg.norm(left_corner - right_corner)

            # Vertical distance (openness of the mouth)
            upper_mean = upper_inner[['x_scaled', 'y_scaled']].mean().values
            lower_mean = lower_inner[['x_scaled', 'y_scaled']].mean().values
            vertical_distance = np.linalg.norm(upper_mean - lower_mean)

            # Combined smile intensity
            smile_intensity = horizontal_distance + vertical_distance
            smile_intensities.append(smile_intensity)
            
        
        if not upper_outer.empty and not lower_outer.empty:
            mar = mouth_aspect_ratio(upper_outer, lower_outer)
            
            if mar > yawn_threshold:
                if not is_yawning:
                    is_yawning = True
                    yawn_start = frame
            else:
                if is_yawning:
                    is_yawning = False
                    yawn_durations.append(frame - yawn_start)
                    yawn_timestamps.append(frame/27)
                    
                    
        # Calculate y-axis tilt (vertical spread)
        if len(y_coords) > 1:
            y_tilt = max(y_coords) - min(y_coords)
            tilts_y.append(y_tilt)

        # Calculate x-axis tilt (horizontal spread)
        if len(x_coords) > 1:
            x_tilt = max(x_coords) - min(x_coords)
            tilts_x.append(x_tilt)
    
    
    if blink_durations:
        metrics["blink_count"] = len(blink_durations)
        metrics["mean_blink_duration"] = np.mean(blink_durations)
        metrics["min_blink_duration"] = np.min(blink_durations)
        metrics["max_blink_duration"] = np.max(blink_durations)
        metrics["std_blink_duration"] = np.std(blink_durations)

    
    metrics["mean_left_eye_openness"] = np.nanmean(eye_openness_per_frame["left_eye_openness"])
    metrics["mean_right_eye_openness"] = np.nanmean(eye_openness_per_frame["right_eye_openness"])

    if smile_intensities:
        metrics["average_smile_intensity"] = np.mean(smile_intensities)
        metrics["max_smile_intensity"] = np.max(smile_intensities)



    nose_data = data[data['landmark_index'] == nose_tip_index].sort_values('frame')

    if not nose_data.empty:
        # Calculate frame-to-frame differences
        horizontal_diff = np.abs(np.diff(nose_data['x_scaled'], prepend=nose_data['x_scaled'].iloc[0]))
        vertical_diff = np.abs(np.diff(nose_data['y_scaled'], prepend=nose_data['y_scaled'].iloc[0]))

        # Sum up the movements
        metrics["total_horizontal_movement"] = np.sum(horizontal_diff)
        metrics["total_vertical_movement"] = np.sum(vertical_diff)
        

    if yawn_durations:
        metrics["yawn_count"] = len(yawn_durations)
        metrics["mean_yawn_duration"] = np.mean(yawn_durations)
        metrics["max_yawn_duration"] = np.max(yawn_durations)
        metrics["std_yawn_duration"] = np.std(yawn_durations)


    # Calculate metrics for y-axis tilt
    if tilts_y:
        metrics["y_tilt_mean"] = np.mean(tilts_y)
        metrics["y_tilt_max"] = np.max(tilts_y)
        metrics["y_tilt_std"] = np.std(tilts_y)

    # Calculate metrics for x-axis tilt
    if tilts_x:
        metrics["x_tilt_mean"] = np.mean(tilts_x)
        metrics["x_tilt_max"] = np.max(tilts_x)
        metrics["x_tilt_std"] = np.std(tilts_x)
    

    return [metrics, blink_durations, blink_timestamps, eye_openness_per_frame, smile_intensities, yawn_durations, yawn_timestamps, horizontal_diff, vertical_diff, tilts_y, tilts_x]


In [38]:
output = calculate_video_metrics_with_tracking(pd.read_csv("Data/face_landmarks/F1/Preprocessed/1_01_0.csv"))

Starting metric calculation process.
0 minute 0.0
1620 minute 1.0
3240 minute 2.0
4860 minute 3.0
6480 minute 4.0
8100 minute 5.0
9720 minute 6.0
11340 minute 7.0
12960 minute 8.0


In [40]:
metrics, blink_durations, blink_timestamps, eye_openness_per_frame, smile_intensities, yawn_durations, yawn_timestamps, horizontal_diff, vertical_diff, tilts_y, tilts_x = output

In [27]:
max_frames = 500*27
is_blinking = [0]*max_frames
for i in range(len(blink_timestamps)):
    for j in range(blink_durations[i]):
        is_blinking[round(blink_timestamps[i]*27)-j] = 1

is_yawning = [0]*max_frames
for i in range(len(yawn_timestamps)):
    for j in range(yawn_durations[i]):
        is_yawning[round(yawn_timestamps[i]*27)-j] = 1
        
        
cumulative_blinks = []
blink_count = 0
blink_in_progress = False

for blink in is_blinking:
    if blink and not blink_in_progress:
        blink_count += 1
        blink_in_progress = True
    elif not blink:
        blink_in_progress = False
    cumulative_blinks.append(blink_count)
    
    
cumulative_yawns = []
yawn_count = 0
yawn_in_progress = False

for yawn in is_yawning:
    if yawn and not yawn_in_progress:
        yawn_count += 1
        yawn_in_progress = True
    elif not yawn:
        yawn_in_progress = False
    cumulative_yawns.append(yawn_count)

In [65]:
len(list(horizontal_diff))

13501

In [4]:
import sys

In [59]:
def overlay_landmarks_and_metrics(video_path, original_landmarks_path, preprocessed_landmark_path, output_path, frame_start=0, frame_end=sys.maxsize):
    """
    Overlays facial landmarks and real-time metrics on a video.

    Parameters:
        video_path (str): Path to the video file.
        original_landmarks (pd.DataFrame): DataFrame with columns ['frame', 'landmark_index', 'x', 'y'].
        scaled_landmarks (pd.DataFrame): DataFrame with columns ['frame', 'landmark_index', 'x_scaled', 'y_scaled'].
        original_fps (float): Original video frames per second.
        scaled_fps (float): FPS of the scaled landmarks dataset.
        output_path (str): Path to save the output video.
    """
    
    def change_fps_list(ls, target_fps, original_fps=27):
        original_length = len(ls)
        target_length = int((len(ls)/original_fps)*target_fps)
        target_ls = []
        for i in range(target_length):
            target_ls.append(ls[int((i/target_fps)*original_fps)])

        return target_ls
    
    original_landmarks = pd.read_csv(original_landmarks_path)
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    fourcc = cv2.VideoWriter_fourcc(*"XVID")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
#     output = calculate_video_metrics_with_tracking(pd.read_csv(preprocessed_landmark_path))
    metrics, blink_durations, blink_timestamps, eye_openness_per_frame, smile_intensities, yawn_durations, yawn_timestamps, horizontal_diff, vertical_diff, tilts_y, tilts_x = output
    
    print("calculating metrics")
    
    max_frames = 500*27
    is_blinking = [0]*max_frames
    for i in range(len(blink_timestamps)):
        for j in range(blink_durations[i]):
            is_blinking[round(blink_timestamps[i]*27)-j] = 1

    is_yawning = [0]*max_frames
    for i in range(len(yawn_timestamps)):
        for j in range(yawn_durations[i]):
            is_yawning[round(yawn_timestamps[i]*27)-j] = 1


    cumulative_blinks = []
    blink_count = 0
    blink_in_progress = False

    for blink in is_blinking:
        if blink and not blink_in_progress:
            blink_count += 1
            blink_in_progress = True
        elif not blink:
            blink_in_progress = False
        cumulative_blinks.append(blink_count)


    cumulative_yawns = []
    yawn_count = 0
    yawn_in_progress = False

    for yawn in is_yawning:
        if yawn and not yawn_in_progress:
            yawn_count += 1
            yawn_in_progress = True
        elif not yawn:
            yawn_in_progress = False
        cumulative_yawns.append(yawn_count)
        
    
    print(f"changing fps from {fps}")
    
    is_blinking_fpschanged = change_fps_list(is_blinking, fps)
    is_yawning_fpschanged = change_fps_list(is_yawning, fps)
    cumulative_blinks_fpschanged = change_fps_list(cumulative_blinks, fps)
    cumulative_yawns_fpschanged = change_fps_list(cumulative_yawns, fps)
    smile_intensities_fpschanged = change_fps_list(smile_intensities, fps)
    horizontal_diff_fpschanged = change_fps_list(horizontal_diff, fps)
    vertical_diff_fpschanged = change_fps_list(vertical_diff, fps)
    tilts_y_fpschanged = change_fps_list(tilts_y, fps)
    tilts_x_fpschanged = change_fps_list(tilts_x, fps)
    print(len(tilts_x_fpschanged))
    
    print("Iterate through video frames")
    
    frame_idx = 0
    while cap.isOpened():
        try:
            frame_idx += 1
            if frame_idx < frame_start:
                continue
            if frame_idx > min(len(is_blinking_fpschanged)-2, frame_end-2, max_frames-2):
                break
            
            ret, frame = cap.read()
            if not ret:
                break

            # Draw original landmarks
            current_landmarks = original_landmarks[original_landmarks['frame'] == frame_idx]
            for _, landmark in current_landmarks.iterrows():
                x, y = int(landmark['x'] * width), int(landmark['y'] * height)
                cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)


            # Get corresponding metrics for this frame
            blink_status = is_blinking_fpschanged[frame_idx]
            yawn_status = is_yawning_fpschanged[frame_idx]
            cumulative_blinks_status = cumulative_blinks_fpschanged[frame_idx]
            cumulative_yawns_status = cumulative_yawns_fpschanged[frame_idx]
            smile_intensity = smile_intensities_fpschanged[frame_idx]
            horizontal_diff_status = horizontal_diff_fpschanged[frame_idx]
            vertical_diff_status = vertical_diff_fpschanged[frame_idx]
            tilt_y_status = tilts_y_fpschanged[frame_idx]
            tilt_x_status = tilts_x_fpschanged[frame_idx]

            # Overlay metrics on the frame
            cv2.putText(frame, f"Blinking: {blink_status}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, f"Yawning: {yawn_status}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, f"Cumulative Blinks: {cumulative_blinks_status}", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, f"Cumulative Yawns: {cumulative_yawns_status}", (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, f"Smile Intensity: {smile_intensity:.2f}", (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, f"Horizontal Diff: {horizontal_diff_status:.2f}", (10, 180), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, f"Vertical Diff: {vertical_diff_status:.2f}", (10, 210), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, f"Tilt Y: {tilt_y_status:.2f}", (10, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, f"Tilt X: {tilt_x_status:.2f}", (10, 270), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

            # Write the frame with overlay to the output video
            out.write(frame)
            
        except:
            print("breaking on frame")
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()


In [7]:
output = calculate_video_metrics_with_tracking(pd.read_csv("Data/face_landmarks/F3/Preprocessed/3_29_10.csv"))

Starting metric calculation process.
0 minute 0.0
1620 minute 1.0
3240 minute 2.0
4860 minute 3.0
6480 minute 4.0
8100 minute 5.0
9720 minute 6.0
11340 minute 7.0
12960 minute 8.0


In [12]:
metrics, blink_durations, blink_timestamps, eye_openness_per_frame, smile_intensities, yawn_durations, yawn_timestamps, horizontal_diff, vertical_diff, tilts_y, tilts_x = output

In [61]:
overlay_landmarks_and_metrics("Data/Raw/3_29_10.mp4", 
                              "Data/face_landmarks/F3/Raw/3_29_10.csv", 
                              "Data/face_landmarks/F3/Preprocessed/3_29_10.csv", 
                              "3_29_10_full.mp4", 0, 8330)

calculating metrics
changing fps from 16.66561152014674
8333
Iterate through video frames
